In [ ]:
%load_ext autoreload
%autoreload 2

import sys 
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_from_mysql
from pyspark.sql import functions as F
from pyspark.sql import Window

cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

fact_transactions = read_from_mysql(
    spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], "fact_transactions"
)

In [ ]:
from pyspark.sql.functions import when


w_naive = Window.partitionBy("account_id").orderBy("date_key", "trans_id")

with_running = (fact_transactions
    .withColumn("signed_amount",
                F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("computed_balance",
                F.sum("signed_amount").over(w_naive))            
)

balance_mismatches = with_running.filter(
    F.abs(F.col("computed_balance") - F.col("balance")) > F.lit(0.01)
)

print("Mismatched rows:", balance_mismatches.count())

## Step 1 — is the failure cascading from a single point per account?

In [ ]:
first_fail_per_account = (balance_mismatches
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("account_id").orderBy("date_key", "trans_id")
    ))
    .filter(F.col("rn") == 1))

print("Accounts with at least one difference:", first_fail_per_account.count())
first_fail_per_account.select(
    "account_id", "date_key", "trans_id", "type", "amount", "balance", "computed_balance"
).show(10)

## Step 2 — pairwise check: does the mismatch depend on cumulative history?

In [ ]:
w = Window.partitionBy("account_id").orderBy("date_key", "trans_id")

pairwise_check = (fact_transactions
    .withColumn("signed_amount",
                F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("prev_balance", F.lag("balance").over(w))
    .withColumn("expected_balance", F.col("prev_balance") + F.col("signed_amount"))
    .withColumn("pairwise_diff", F.round(F.col("balance") - F.col("expected_balance")))
)

pairwise_check.filter(F.col("pairwise_diff") != 0).groupBy("pairwise_diff").count().orderBy(F.desc("count")).show(20)

## Step 3 — are same-day multi-transaction combinations actually common?

In [ ]:
smae_day_multi = (
    fact_transactions
    .groupBy("account_id", "date_key")
    .count()
    .filter(F.col("count") > 1)
)

print("Account+day combinations with multiple transactions:", smae_day_multi.count())

## Step 4 — check at the day level instead of the transaction level

In [ ]:
last_of_day = (
    fact_transactions
    .withColumn("signed_amount", F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("rn_desc", F.row_number().over(
        Window.partitionBy("account_id", "date_key").orderBy(F.desc("trans_id"))))
)

daily_summary = (last_of_day
    .groupBy("account_id", "date_key")
    .agg(F.sum("signed_amount").alias("daily_net_change")))

daily_end_balance = (last_of_day
    .filter(F.col("rn_desc") == 1)
    .select("account_id", "date_key", F.col("balance").alias("last_balance_of_day")))

daily_check = (daily_summary
    .join(daily_end_balance, ["account_id", "date_key"])
    .withColumn("prev_day_balance", F.lag("last_balance_of_day").over(
        Window.partitionBy("account_id").orderBy("date_key")))
    .withColumn("day_diff", F.round(
        F.col("last_balance_of_day") - (F.col("prev_day_balance") + F.col("daily_net_change")), 2))
)

daily_check.filter(F.col("day_diff") != 0).count()    

In [ ]:
single_txn_days = (fact_transactions
    .groupBy("account_id", "date_key")
    .count()
    .filter(F.col("count") == 1)
    .select("account_id", "date_key"))

single_day_failures = (daily_check
    .join(single_txn_days, ["account_id", "date_key"])
    .filter(F.col("day_diff") != 0))

print("Single-transaction days with a mismatch:", single_day_failures.count())

## Step 5 — test the "sign flip" hypothesis on single-transaction days

In [ ]:
sign_flip_check = (single_day_failures
    .withColumn("is_sign_flip", F.col("day_diff") == -2 * F.col("daily_net_change")))

sign_flip_check.groupBy("is_sign_flip").count().show()

(sign_flip_check
    .filter(F.col("is_sign_flip"))
    .join(fact_transactions, ["account_id", "date_key"])
    .groupBy("type")
    .count()
    .show())

## Step 6 — manual verification in DBeaver (SQL, executed outside this notebook)

Found via account_id=1033, day 980630:
- trans_id 303,570 (PRIJEM, VKLAD, 6,463) and trans_id 3,440,301
  (PRIJEM, k_symbol=UROK, 163.8) — same day.
- Ordering by trans_id gives no match against either stored balance.
- Ordering UROK first: 30,432.2 + 163.8 = 30,596.0 (matches UROK row's
  balance 30,595.9 within the known 0.1 rounding tolerance), then
  30,595.9 + 6,463 = 37,058.9 — exact match to the VKLAD row's balance.

Root cause: UROK trans_id values are in a completely different numeric
range (~3,440,000s) than regular transactions for the same account
(~300,000s) — confirming UROK rows were inserted by a separate batch
process, not in real-time chronological order.

## Step 7 — verify the fix at full scale

In [ ]:
from src.validations import check_running_balance

balance_result_v2, balance_mismatches_v2 = check_running_balance(fact_transactions)
print(balance_result_v2)

In [ ]:
remaining = balance_mismatches_v2.withColumn(
    "diff", F.round(F.col("computed_balance") - F.col("balance"), 2)
)
remaining.groupBy("diff").count().orderBy(F.desc("count")).show(10)
remaining.groupBy((F.col("k_symbol") == "UROK")).count().show()

In [ ]:
w_fixed = Window.partitionBy("account_id").orderBy(
    "date_key",
    F.when(F.col("k_symbol") == "UROK", F.lit(0)).otherwise(F.lit(1)),
    "trans_id"
)

pairwise_fixed = (fact_transactions
    .withColumn("signed_amount", F.when(F.col("type") == "PRIJEM", F.col("amount")).otherwise(-F.col("amount")))
    .withColumn("prev_balance", F.lag("balance").over(w_fixed))
    .withColumn("expected_balance", F.col("prev_balance") + F.col("signed_amount"))
    .withColumn("pairwise_diff", F.round(F.col("balance") - F.col("expected_balance"), 2))
)

pairwise_fixed.filter(F.col("pairwise_diff") != 0).groupBy(F.col("k_symbol") == "UROK").count().show()